<a href="https://colab.research.google.com/github/rwalden-gif/IntroToML/blob/main/Homework6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import	torch
import	torch.nn	as	nn
import	pandas	as	pd
import	matplotlib.pyplot	as	plt
from	sklearn.model_selection	import	train_test_split
from	sklearn.preprocessing	import	StandardScaler
from	sklearn.linear_model	import	LinearRegression,	LogisticRegression
from	sklearn.svm	import	SVR,	SVC
from	sklearn.metrics	import	mean_squared_error,	r2_score
from	sklearn.metrics	import	accuracy_score,	precision_score,	recall_score,	f1_score
from	sklearn.metrics	import	ConfusionMatrixDisplay
housing_url	=	"https://raw.githubusercontent.com/rwalden-gif/IntroToML/main/Housing.csv"
cancer_url	=	"https://raw.githubusercontent.com/rwalden-gif/IntroToML/main/Cancer.csv"

In [ ]:
data	=	pd.read_csv(housing_url)
for	col	in	["mainroad",	"guestroom",	"basement",	"hotwaterheating",	"airconditioning",	"prefarea"]:
data[col]	=	(data[col]	==	"yes").astype(int)
data	=	pd.get_dummies(data,	columns=["furnishingstatus"],	drop_first=True,	dtype=int)
data	=	data.sample(frac=1,	random_state=42)
split	=	int(0.8	*	len(data))
train	=	data[:split]
test	=	data[split:]
scaler	=	StandardScaler().fit(train.drop(columns="price"))
X_train	=	torch.tensor(scaler.transform(train.drop(columns="price")),	dtype=torch.float32)
X_test	=	torch.tensor(scaler.transform(test.drop(columns="price")),	dtype=torch.float32)
y_train	=	torch.tensor(train["price"].values	/	1000000,	dtype=torch.float32)
y_test	=	torch.tensor(test["price"].values	/	1000000,	dtype=torch.float32)
print("Inputs:",	X_train.shape[1])
print("Training	rows:",	len(X_train),	"Test	rows:",	len(X_test))

In [ ]:
def	make_net(n_inputs,	hidden):
  return	nn.Sequential(
    nn.Linear(n_inputs,	hidden),	nn.ReLU(),
    nn.Linear(hidden,	hidden),	nn.ReLU(),
    nn.Linear(hidden,	1))

def	train_housing(hidden,	lr,	epochs,	X_a,	y_a,	X_b,	y_b):
  torch.manual_seed(42)
  net	=	make_net(X_a.shape[1],	hidden)
  optimizer	=	torch.optim.Adam(net.parameters(),	lr=lr)
  loss_fn	=	nn.MSELoss()
  train_losses,	val_losses	=	[],	[]
  for	epoch	in	range(epochs):
    loss	=	loss_fn(net(X_a).squeeze(1),	y_a)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    with	torch.no_grad():
      train_losses.append(loss_fn(net(X_a).squeeze(1),	y_a).item())
      val_losses.append(loss_fn(net(X_b).squeeze(1),	y_b).item())
  return	net,	train_losses,	val_losses

In [ ]:
cut	=	int(0.8	*	len(X_train))
best_loss	=	1e9
for	hidden	in	[8,	16,	32]:
  for	lr	in	[0.01,	0.001]:
    net,	train_losses,	val_losses	=	train_housing(hidden,	lr,	100,	X_train[:cut],	y_train[:cut],	X_train[cut:],	y_train[cut:])
    print("Neurons	per	hidden	layer",	hidden,	"learning	rate",	lr,	"validation	loss",	round(val_losses[-1],	4))
    if	val_losses[-1]	<	best_loss:
      best_loss	=	val_losses[-1]
      best_hidden	=	hidden
      best_lr	=	lr
print()
print("Chosen	neurons	per	hidden	layer:",	best_hidden)
print("Chosen	learning	rate:",	best_lr)

In [ ]:
net,	train_losses,	val_losses	=	train_housing(best_hidden,	best_lr,	100,	X_train,	y_train,	X_test,	y_test)
plt.figure(figsize=(7,	5))
plt.plot(train_losses,	label="Training	loss")
plt.plot(val_losses,	label="Validation	loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Housing	Neural	Network	Loss")
plt.legend()
plt.show()

In [ ]:
nn_pred	=	net(X_test).squeeze(1).detach().numpy()	*	1000000
true_price	=	y_test.numpy()	*	1000000

mse	=	mean_squared_error(true_price,	nn_pred)
print("Neural	network	MSE:",	mse)
print("Neural	network	RMSE:",	mse	**	0.5)
print("Neural	network	R2:",	r2_score(true_price,	nn_pred))

In [ ]:
linear	=	LinearRegression().fit(X_train,	y_train)
svr	=	SVR().fit(X_train,	y_train)

linear_pred	=	linear.predict(X_test)	*	1000000
svr_pred	=	svr.predict(X_test)	*	1000000

print("Model															MSE												RMSE									R2")
for	name,	pred	in	[("Neural	network",	nn_pred),	("Linear	regression",	linear_pred),	("SVR",	svr_pred)]:
    mse	=	mean_squared_error(true_price,	pred)
    print(f"{name:18s}	{mse:.4e}			{mse	**	0.5:,.0f}			{r2_score(true_price,	pred):.4f}")

In [ ]:
cancer	=	pd.read_csv(cancer_url)
cancer	=	cancer.drop(columns=["id",	"Unnamed:	32"])
labels	=	(cancer["diagnosis"]	==	"M").astype(int).values
features	=	cancer.drop(columns="diagnosis").values

X_train_raw,	X_test_raw,	y_train_c,	y_test_c	=	train_test_split(features,	labels,	test_size=0.2,	random_state=42,	stratify=labels)

scaler	=	StandardScaler().fit(X_train_raw)
X_train_c	=	torch.tensor(scaler.transform(X_train_raw),	dtype=torch.float32)
X_test_c	=	torch.tensor(scaler.transform(X_test_raw),	dtype=torch.float32)
y_train_t	=	torch.tensor(y_train_c,	dtype=torch.float32)
y_test_t	=	torch.tensor(y_test_c,	dtype=torch.float32)

print("Inputs:",	X_train_c.shape[1])
print("Training	rows:",	len(X_train_c),	"Test	rows:",	len(X_test_c))

In [ ]:
def	train_cancer(hidden,	lr,	epochs,	X_a,	y_a,	X_b,	y_b):
				torch.manual_seed(42)
				net	=	make_net(X_a.shape[1],	hidden)
				optimizer	=	torch.optim.Adam(net.parameters(),	lr=lr)
				loss_fn	=	nn.BCEWithLogitsLoss()
				train_losses,	val_losses,	train_accs,	val_accs	=	[],	[],	[],	[]
				for	epoch	in	range(epochs):
								loss	=	loss_fn(net(X_a).squeeze(1),	y_a)
								optimizer.zero_grad()
								loss.backward()
								optimizer.step()
								with	torch.no_grad():
												out_a	=	net(X_a).squeeze(1)
												out_b	=	net(X_b).squeeze(1)
												train_losses.append(loss_fn(out_a,	y_a).item())
												val_losses.append(loss_fn(out_b,	y_b).item())
												train_accs.append(((out_a	>	0).float()	==	y_a).float().mean().item())
												val_accs.append(((out_b	>	0).float()	==	y_b).float().mean().item())
				return	net,	train_losses,	val_losses,	train_accs,	val_accs

In [ ]:
cut	=	int(0.8	*	len(X_train_c))
best_loss	=	1e9
for	hidden	in	[8,	16,	32]:
				for	lr	in	[0.01,	0.001]:
								net,	train_losses,	val_losses,	train_accs,	val_accs	=	train_cancer(hidden,	lr,	100,	X_train_c[:cut],	y_train_t[:cut],	X_tra
in_c[cut:],	y_train_t[cut:])
								print("Neurons	per	hidden	layer",	hidden,	"learning	rate",	lr,	"validation	loss",	round(val_losses[-1],	4))
								if	val_losses[-1]	<	best_loss:
												best_loss	=	val_losses[-1]
												best_hidden_c	=	hidden
												best_lr_c	=	lr
print()
print("Chosen	neurons	per	hidden	layer:",	best_hidden_c)
print("Chosen	learning	rate:",	best_lr_c)

In [ ]:
net_c,	train_losses,	val_losses,	train_accs,	val_accs	=	train_cancer(best_hidden_c,	best_lr_c,	100,	X_train_c,	y_train_t,	X_test_c,
y_test_t)

plt.figure(figsize=(7,	5))
plt.plot(train_losses,	label="Training	loss")
plt.plot(val_losses,	label="Validation	loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Cancer	Neural	Network	Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7,	5))
plt.plot(train_accs,	label="Training	accuracy")
plt.plot(val_accs,	label="Validation	accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Cancer	Neural	Network	Accuracy")
plt.legend()
plt.show()

In [ ]:
nn_labels	=	(net_c(X_test_c).squeeze(1)	>	0).int().numpy()

print("Neural	network	accuracy:",	accuracy_score(y_test_c,	nn_labels))
print("Neural	network	precision:",	precision_score(y_test_c,	nn_labels))
print("Neural	network	recall:",	recall_score(y_test_c,	nn_labels))
print("Neural	network	F1:",	f1_score(y_test_c,	nn_labels))

ConfusionMatrixDisplay.from_predictions(y_test_c,	nn_labels,	display_labels=["Benign",	"Malignant"],	cmap="Blues")
plt.title("Cancer	Neural	Network	Confusion	Matrix")
plt.show()

In [ ]:
logistic	=	LogisticRegression(max_iter=1000).fit(X_train_c,	y_train_c)
svm	=	SVC().fit(X_train_c,	y_train_c)

logistic_labels	=	logistic.predict(X_test_c)
svm_labels	=	svm.predict(X_test_c)

print("Model																Accuracy		Precision		Recall			F1")
for	name,	pred	in	[("Neural	network",	nn_labels),	("Logistic	regression",	logistic_labels),	("SVM",	svm_labels)]:
  print(f"{name:19s}		{accuracy_score(y_test_c,	pred):.4f}
red):.4f}			{f1_score(y_test_c,	pred):.4f}")